# Data Download Pipeline
## Neural PK-PD Modeling with Physics-Informed Neural ODEs

**Purpose**: Consolidated notebook for downloading and validating all project datasets from 5 sources.

| Section | Source | Data Type | Records |
|---------|--------|-----------|---------|
| 1 | Setup | Imports, config, utilities | — |
| 2 | PubChem | hERG & CYP3A4 assays, exemplar compounds | ~5,000 |
| 3 | PK-DB | Studies metadata, PK parameters, timecourses | 796 studies |
| 4 | TDC ADMET | Caco-2, hERG, clearance, solubility, etc. | ~8,600 |
| 5 | ChEMBL | Binding affinity (synthetic + real SMILES) | ~5,400 |
| 6 | ToxCast/Tox21 | Toxicity screening (7 categories) | ~50,000 |
| 7 | Summary | Verification & file inventory | — |
| 8 | Timeline | Chronological execution log | — |

**Run order**: Execute cells top to bottom. Each section is independent — skip any source you don't need.  
**Timing**: Every code cell logs its start/end time. Run the final cell (§ 8) to see the full chronological execution timeline.

> This notebook consolidates the former standalone scripts: `data_download.py`, `download_chembl_binding_data.py`, `download_chembl_binding_real.py`, `download_pkdb_complete.py`, `download_tdc_admet.py`, `download_toxcast_tox21.py`, `explore_pkdb_data.py`.

## Section 1: Setup — Imports, Configuration & Utility Functions

In [ ]:
"""Section 1A: Imports — all libraries used across the download pipeline."""
from __future__ import annotations

import json
import time
import warnings
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests

# Optional: RDKit for molecular validation
try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors
    HAS_RDKIT = True
except ImportError:
    HAS_RDKIT = False
    print("⚠  RDKit not available — MW/LogP will be NaN for ChEMBL real-SMILES")

# Optional: ChEMBL Python client
try:
    from chembl_webresource_client.new_client import new_client
    CHEMBL_CLIENT_AVAILABLE = True
except ImportError:
    CHEMBL_CLIENT_AVAILABLE = False

warnings.filterwarnings("ignore")

# ── Cell Execution Tracker ──────────────────────────────────────────────
CELL_EXEC_LOG: list[dict] = []


def log_cell_start(section: str) -> None:
    """Record cell execution start time."""
    now = datetime.now()
    CELL_EXEC_LOG.append({"section": section, "start": now, "end": None, "duration_s": None})
    print(f"⏱  [{now.strftime('%Y-%m-%d %H:%M:%S')}] Starting: {section}")


def log_cell_end() -> None:
    """Record cell execution end time for the most recent entry."""
    now = datetime.now()
    if CELL_EXEC_LOG and CELL_EXEC_LOG[-1]["end"] is None:
        entry = CELL_EXEC_LOG[-1]
        entry["end"] = now
        entry["duration_s"] = round((now - entry["start"]).total_seconds(), 2)
        print(f"⏱  [{now.strftime('%Y-%m-%d %H:%M:%S')}] Completed in {entry['duration_s']:.1f}s")

# ────────────────────────────────────────────────────────────────────────

log_cell_start("§1A  Imports")

print("✓ All imports loaded")
print(f"  RDKit available: {HAS_RDKIT}")
print(f"  ChEMBL client available: {CHEMBL_CLIENT_AVAILABLE}")

log_cell_end()

In [ ]:
"""Section 1B: Configuration — paths, API endpoints, constants."""
log_cell_start("§1B  Configuration")

# Root directory (Coding/)
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data" / "raw"

# Per-source subdirectories
PUBCHEM_DIR = DATA_DIR / "pubchem"
PKDB_DIR    = DATA_DIR / "pkdb"
TDC_DIR     = DATA_DIR / "tdc"
CHEMBL_DIR  = DATA_DIR / "chembl"
TOXCAST_DIR = DATA_DIR / "toxcast"

# API endpoints
PUBCHEM_API  = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
PKDB_API     = "https://pk-db.com/api/v1"
CHEMBL_API   = "https://www.ebi.ac.uk/chembl/api/data"
CHEMBL_ACT   = f"{CHEMBL_API}/activity"

# Download settings
REQUEST_TIMEOUT = 60       # seconds
CHEMBL_DELAY    = 0.35     # seconds between ChEMBL API pages
CHEMBL_PAGE     = 1000     # records per ChEMBL API page
MAX_PER_TARGET  = 600      # cap per ChEMBL target for diversity

# ChEMBL real-SMILES targets (Section 5B)
CHEMBL_REAL_TARGETS = [
    ("CHEMBL217",  "Dopamine D2 receptor"),
    ("CHEMBL251",  "Adenosine A2a receptor"),
    ("CHEMBL203",  "EGFR"),
    ("CHEMBL301",  "CDK2"),
    ("CHEMBL210",  "Beta-2 adrenergic receptor"),
    ("CHEMBL1871", "Androgen receptor"),
    ("CHEMBL2034", "Glucocorticoid receptor"),
    ("CHEMBL325",  "Serotonin 5-HT2A receptor"),
]

print(f"✓ Configuration loaded")
print(f"  Data root: {DATA_DIR}")
print(f"  Subdirs: pubchem, pkdb, tdc, chembl, toxcast")

log_cell_end()

In [ ]:
"""Section 1C: Utility functions — shared helpers for all download sections."""
log_cell_start("§1C  Utilities")


def ensure_dir(path: Path) -> None:
    """Create directory (and parents) if it doesn't exist."""
    path.mkdir(parents=True, exist_ok=True)


def download_file(url: str, dest: Path, *, overwrite: bool = False) -> Optional[Path]:
    """Download a single file with streaming. Returns path on success, None on failure."""
    if dest.exists() and not overwrite:
        print(f"  [skip] {dest.name} exists")
        return dest
    try:
        with requests.get(url, stream=True, timeout=REQUEST_TIMEOUT) as r:
            r.raise_for_status()
            ensure_dir(dest.parent)
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        print(f"  [ok] {dest.name}")
        return dest
    except Exception as e:
        print(f"  [fail] {dest.name}: {e}")
        return None


def api_get(url: str, params: dict = None, headers: dict = None, retries: int = 3) -> Optional[dict]:
    """GET JSON from an API with retry logic."""
    for attempt in range(retries):
        try:
            r = requests.get(
                url,
                params=params,
                headers=headers or {"Accept": "application/json"},
                timeout=REQUEST_TIMEOUT,
            )
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt == retries - 1:
                print(f"  ⚠ GET failed after {retries} attempts: {e}")
                return None
            time.sleep(1.5)


def file_summary(directory: Path) -> pd.DataFrame:
    """Return a DataFrame listing all files in a directory with sizes."""
    if not directory.exists():
        return pd.DataFrame(columns=["file", "size_kb"])
    rows = []
    for f in sorted(directory.rglob("*")):
        if f.is_file():
            rows.append({"file": str(f.relative_to(directory)), "size_kb": round(f.stat().st_size / 1024, 1)})
    return pd.DataFrame(rows)


# Create all data directories
for d in [PUBCHEM_DIR, PKDB_DIR, TDC_DIR, CHEMBL_DIR, TOXCAST_DIR]:
    ensure_dir(d)

print("✓ Utility functions defined")
print("✓ Data directories created")

log_cell_end()

## Section 2: PubChem — Assays & Exemplar Compounds

Downloads from the PubChem PUG REST API:
- **hERG qHTS assay** (AID 588834) — cardiac safety screening
- **CYP3A4 inhibition assay** (AID 54772) — metabolic interaction
- **Exemplar compounds** (warfarin, midazolam, caffeine) — 3D SDF structures

> Source: https://pubchem.ncbi.nlm.nih.gov/

In [ ]:
"""Section 2: PubChem downloads — assays + exemplar compounds."""
log_cell_start("§2   PubChem Downloads")


def fetch_pubchem_assay(aid: int, label: str) -> Optional[Path]:
    """Download a PubChem bioassay as CSV."""
    url = f"{PUBCHEM_API}/assay/aid/{aid}/CSV"
    dest = PUBCHEM_DIR / f"assay_{label}_aid{aid}.csv"
    return download_file(url, dest)


def fetch_pubchem_compound(name: str) -> Optional[Path]:
    """Download a PubChem compound as 3D SDF."""
    url = f"{PUBCHEM_API}/compound/name/{name}/SDF?record_type=3d"
    dest = PUBCHEM_DIR / f"compound_{name.lower()}.sdf"
    return download_file(url, dest)


# --- Execute PubChem Downloads ---
print("=" * 60)
print("PUBCHEM DATA DOWNLOAD")
print("=" * 60)

# Bioassays
print("\n📥 Bioassays:")
fetch_pubchem_assay(588834, label="herg_qhts")
fetch_pubchem_assay(54772, label="cyp3a4_inhibition")

# Exemplar compounds
print("\n📥 Exemplar compounds (3D SDF):")
for compound in ["warfarin", "midazolam", "caffeine"]:
    fetch_pubchem_compound(compound)

# Summary
print("\n📁 PubChem files:")
display(file_summary(PUBCHEM_DIR))

log_cell_end()

## Section 3: PK-DB — Pharmacokinetic Studies & Parameters

Downloads from the PK-DB REST API:
- **Studies metadata** — 796 studies with drug names, subjects, PK parameters
- **PK parameter outputs** — CL, Vd, t1/2, AUC, Cmax, etc. (by ID from studies)
- **Timecourse subsets** — time-concentration curve containers

> Source: https://www.pk-db.com/  
> Note: Some endpoints require authentication. Timecourse data may be limited.

In [ ]:
"""Section 3A: PK-DB download functions."""
log_cell_start("§3A  PK-DB Functions")


def fetch_pkdb_studies(limit: int = 100) -> Optional[list]:
    """Fetch PK-DB studies metadata. Returns list of study dicts."""
    url = f"{PKDB_API}/studies/?format=json"
    dest_full = PKDB_DIR / "studies.json"
    dest_trim = PKDB_DIR / f"studies_top{limit}.json"

    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        dest_full.write_bytes(r.content)
        print(f"  [ok] studies.json (full)")

        data = r.json()
        # Handle nested structure: data.data.data
        if "data" in data and isinstance(data["data"], dict):
            studies = data["data"].get("data", [])
        elif "data" in data and isinstance(data["data"], list):
            studies = data["data"]
        else:
            studies = data.get("results", [])

        # Save trimmed version
        trimmed = {"data": studies[:limit]}
        dest_trim.write_text(json.dumps(trimmed, indent=2))
        print(f"  [ok] studies_top{limit}.json ({len(studies[:limit])} studies)")
        return studies
    except Exception as e:
        print(f"  [fail] PK-DB studies: {e}")
        return None


def fetch_pkdb_complete_studies() -> Optional[list]:
    """Download PK-DB studies with full embedded metadata (outputs, subsets, timecourses)."""
    url = f"{PKDB_API}/studies/?format=json"
    dest = PKDB_DIR / "pkdb_studies_complete.json"

    try:
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        data = r.json()

        if "data" in data and isinstance(data["data"], dict):
            studies = data["data"].get("data", [])
        elif "data" in data and isinstance(data["data"], list):
            studies = data["data"]
        else:
            studies = data.get("results", [])

        dest.write_text(json.dumps({"studies": studies}, indent=2))
        print(f"  [ok] pkdb_studies_complete.json ({len(studies)} studies)")

        # Statistics
        total_outputs = sum(len(s.get("outputset", {}).get("outputs", [])) for s in studies)
        total_subsets = sum(len(s.get("dataset", {}).get("subsets", [])) for s in studies)
        total_tc = sum(s.get("timecourse_count", 0) for s in studies)
        print(f"       PK outputs: {total_outputs}, Subsets: {total_subsets}, Timecourses: {total_tc}")
        return studies
    except Exception as e:
        print(f"  [fail] PK-DB complete: {e}")
        return None


def fetch_pkdb_outputs_by_ids(output_ids: list[int], max_fetch: int = 50) -> None:
    """Fetch individual PK parameter outputs by their study-embedded IDs."""
    for i, oid in enumerate(output_ids[:max_fetch]):
        url = f"{PKDB_API}/outputs/{oid}/?format=json"
        dest = PKDB_DIR / f"output_{oid}.json"
        if dest.exists():
            continue
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            dest.write_bytes(r.content)
            if i % 10 == 0:
                print(f"  [ok] output {i + 1}/{min(max_fetch, len(output_ids))}")
        except Exception as e:
            print(f"  [fail] output {oid}: {e}")


def fetch_pkdb_timecourse_subsets(subset_ids: list[int], max_fetch: int = 20) -> None:
    """Fetch timecourse subset containers by their IDs."""
    for i, sid in enumerate(subset_ids[:max_fetch]):
        url = f"{PKDB_API}/subsets/{sid}/?format=json"
        dest = PKDB_DIR / f"subset_{sid}.json"
        if dest.exists():
            continue
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            dest.write_bytes(r.content)
            if i % 5 == 0:
                print(f"  [ok] subset {i + 1}/{min(max_fetch, len(subset_ids))}")
        except Exception as e:
            print(f"  [fail] subset {sid}: {e}")

log_cell_end()

In [ ]:
"""Section 3B: Execute PK-DB downloads."""
log_cell_start("§3B  PK-DB Downloads")

print("=" * 60)
print("PK-DB DATA DOWNLOAD")
print("=" * 60)

# 1. Studies metadata (trimmed + complete)
print("\n📥 Studies metadata:")
studies = fetch_pkdb_studies(limit=100)

print("\n📥 Complete studies (with embedded PK data):")
studies_full = fetch_pkdb_complete_studies()

# 2. Extract output/subset IDs and fetch individual records
if studies_full:
    output_ids = []
    subset_ids = []
    for s in studies_full[:100]:
        output_ids.extend(s.get("outputset", {}).get("outputs", []))
        subset_ids.extend(s.get("dataset", {}).get("subsets", []))
    output_ids = list(set(output_ids))[:100]
    subset_ids = list(set(subset_ids))[:20]

    print(f"\n  Found {len(output_ids)} unique output IDs, {len(subset_ids)} unique subset IDs")

    if output_ids:
        print(f"\n📥 Fetching {min(50, len(output_ids))} PK parameter outputs:")
        fetch_pkdb_outputs_by_ids(output_ids)

    if subset_ids:
        print(f"\n📥 Fetching {min(20, len(subset_ids))} timecourse subsets:")
        fetch_pkdb_timecourse_subsets(subset_ids)

# Summary
print("\n📁 PK-DB files:")
display(file_summary(PKDB_DIR))

log_cell_end()

In [ ]:
"""Section 3C: PK-DB data exploration — structure overview & drug inventory."""
log_cell_start("§3C  PK-DB Exploration")

DATA_FILE = PKDB_DIR / "pkdb_studies_complete.json"

if DATA_FILE.exists():
    with open(DATA_FILE) as f:
        pkdb_data = json.load(f)
    pkdb_studies = pkdb_data["studies"]

    print("=" * 60)
    print("PK-DB DATA EXPLORATION")
    print("=" * 60)

    # Overview
    print(f"\n📊 DATASET OVERVIEW")
    print(f"  Total studies: {len(pkdb_studies)}")
    print(f"  Total subjects: {sum(s.get('individual_count', 0) for s in pkdb_studies)}")
    total_outputs = sum(len(s["outputset"]["outputs"]) for s in pkdb_studies)
    total_tc = sum(s.get("timecourse_count", 0) for s in pkdb_studies)
    print(f"  Total PK outputs: {total_outputs}")
    print(f"  Total timecourses: {total_tc}")

    # Drug inventory
    all_drugs = Counter()
    for s in pkdb_studies:
        for sub in s.get("substances", []):
            all_drugs[sub["name"]] += 1

    print(f"\n💊 DRUGS & SUBSTANCES ({len(all_drugs)} unique)")
    for drug, count in all_drugs.most_common(15):
        print(f"  • {drug:30s} ({count} studies)")

    # Top studies
    print(f"\n📚 TOP 5 STUDIES (by PK outputs)")
    top5 = sorted(pkdb_studies, key=lambda s: len(s["outputset"]["outputs"]), reverse=True)[:5]
    for i, s in enumerate(top5, 1):
        drugs = [sub["name"] for sub in s.get("substances", [])][:3]
        print(f"  {i}. {s['name']}")
        print(f"     Drugs: {', '.join(drugs)} | Outputs: {len(s['outputset']['outputs'])} | TC: {s['timecourse_count']}")

    # Available PK parameters
    print(f"\n📈 AVAILABLE PK PARAMETER TYPES")
    print("  CL, Vd, t½, AUC, Cmax, Tmax, ka, F, MRT, λz, fu, PPB")
else:
    print("⚠ pkdb_studies_complete.json not found — run Section 3B first")

log_cell_end()

## Section 4: TDC — ADMET Benchmark Datasets

Downloads from Therapeutics Data Commons (TDC):
- **Absorption**: Caco-2, HIA, Solubility
- **Distribution**: Lipophilicity, Plasma Protein Binding
- **Metabolism**: Hepatocyte Clearance, Half-Life
- **Toxicity**: hERG Channel Inhibition

> Source: https://tdcommons.ai/  
> Requires: `pip install PyTDC` (the `tdc` package)

In [ ]:
"""Section 4: TDC ADMET benchmark dataset downloads."""
log_cell_start("§4   TDC ADMET Downloads")

# ADMET tasks: (dataset_name, label, category)
TDC_ADMET_TASKS = [
    # Absorption
    ("caco2_wang",              "Caco-2 Cell Permeability",             "Absorption"),
    ("hia_hou",                 "Human Intestinal Absorption",          "Absorption"),
    ("solubility_aqsoldb",      "Aqueous Solubility",                   "Absorption"),
    # Distribution
    ("lipophilicity_astrazeneca", "Lipophilicity (LogP)",               "Distribution"),
    ("ppbr_az",                 "Plasma Protein Binding",               "Distribution"),
    # Metabolism
    ("clearance_hepatocyte_az", "Hepatocyte Clearance",                 "Metabolism"),
    ("half_life_obach",         "Terminal Half-Life",                    "Metabolism"),
    # Toxicity
    ("herg",                    "hERG Channel Inhibition",              "Toxicity"),
]


def download_tdc_dataset(dataset_name: str, label: str) -> Optional[Tuple[pd.DataFrame, dict]]:
    """Download a single TDC ADMET dataset. Returns (DataFrame, metadata) or None."""
    try:
        from tdc.benchmark_group import admet_group

        print(f"  ⏳ {label}...", end="", flush=True)
        group = admet_group(name=dataset_name)
        train_val, test = group.get_train_valid_test(seed=42)
        data = pd.concat([train_val, test], ignore_index=True)

        metadata = {
            "name": dataset_name,
            "label": label,
            "rows": len(data),
            "columns": list(data.columns),
            "train_size": len(train_val),
            "test_size": len(test),
        }
        print(f" ✅ ({len(data)} samples)")
        return data, metadata
    except ImportError:
        print(f" ❌ TDC not installed — run: pip install PyTDC")
        return None
    except Exception as e:
        print(f" ❌ {e}")
        return None


# --- Execute TDC Downloads ---
print("=" * 60)
print("TDC ADMET BENCHMARK DOWNLOADS")
print("=" * 60)

tdc_metadata = {}
categories_seen = set()

for dataset_name, label, category in TDC_ADMET_TASKS:
    if category not in categories_seen:
        print(f"\n📚 {category}")
        categories_seen.add(category)

    result = download_tdc_dataset(dataset_name, label)
    if result is not None:
        data, meta = result
        filepath = TDC_DIR / f"tdc_{dataset_name}.csv"
        data.to_csv(filepath, index=False)
        tdc_metadata[dataset_name] = meta

# Save metadata
if tdc_metadata:
    (TDC_DIR / "tdc_metadata.json").write_text(json.dumps(tdc_metadata, indent=2))
    total = sum(m["rows"] for m in tdc_metadata.values())
    print(f"\n✅ Downloaded {len(tdc_metadata)}/{len(TDC_ADMET_TASKS)} datasets ({total:,} total samples)")

print("\n📁 TDC files:")
display(file_summary(TDC_DIR))

log_cell_end()

## Section 5: ChEMBL — Binding Affinity Data

Two sub-sections:
- **5A**: Representative/synthetic binding data (for rapid prototyping)
- **5B**: Real binding SMILES from ChEMBL REST API (8 protein targets, ~3,400 compounds)

> Source: https://www.ebi.ac.uk/chembl/  
> License: CC Attribution-SA 3.0

In [ ]:
"""Section 5A: ChEMBL representative/synthetic binding affinity generator.

Generates realistic binding data distributions (Ki, IC50, EC50) for rapid
prototyping. Use Section 5B for real data with validated SMILES.
"""
log_cell_start("§5A  ChEMBL Synthetic Binding")


def generate_representative_binding_data(
    n_compounds: int = 2000, n_targets: int = 8, seed: int = 42
) -> Tuple[pd.DataFrame, dict]:
    """Generate representative ChEMBL binding data with realistic distributions."""
    np.random.seed(seed)

    target_templates = [
        ("CHEMBL231", "Histamine H1 receptor"),
        ("CHEMBL218", "Dopamine D2 receptor"),
        ("CHEMBL213", "Serotonin 1a (5-HT1a) receptor"),
        ("CHEMBL228", "Beta-1 adrenergic receptor"),
        ("CHEMBL219", "D1 dopamine receptor"),
        ("CHEMBL205", "Histamine H2 receptor"),
        ("CHEMBL302", "Cyclooxygenase-1"),
        ("CHEMBL388", "Tumor necrosis factor"),
    ]
    targets = target_templates[: min(n_targets, len(target_templates))]
    activity_types = ["Ki", "IC50", "EC50"]

    rows = []
    for idx in range(n_compounds):
        target_id, target_name = targets[idx % len(targets)]
        pchembl = np.random.choice(
            [np.random.normal(5.5, 1.2), np.random.normal(8.5, 0.8)], p=[0.8, 0.2]
        )
        pchembl = np.clip(pchembl, 3.0, 11.0)
        standard_value = 10 ** (-pchembl + 9)

        rows.append({
            "compound_id": f"CHEMBL{1000000 + idx}",
            "target_id": target_id,
            "target_name": target_name,
            "assay_id": f"CHEMBL{1900000 + idx // 10}",
            "standard_type": np.random.choice(activity_types, p=[0.4, 0.4, 0.2]),
            "standard_value": round(standard_value, 2),
            "standard_units": "nM",
            "pchembl_value": round(pchembl, 2),
            "activity_type": "BINDING",
            "source": "ChEMBL",
            "MW": round(np.random.normal(350, 80), 1),
            "LogP": round(np.random.normal(2.5, 1.2), 2),
            "HBA": int(np.random.poisson(3)),
            "HBD": int(np.random.poisson(2)),
        })

    df = pd.DataFrame(rows)
    metadata = {
        "source": "ChEMBL (representative)",
        "n_compounds": len(df["compound_id"].unique()),
        "n_targets": len(df["target_id"].unique()),
        "total_records": len(df),
        "pchembl_range": [float(df["pchembl_value"].min()), float(df["pchembl_value"].max())],
        "license": "CC Attribution-SA 3.0",
    }
    return df, metadata


# --- Generate synthetic binding data ---
print("=" * 60)
print("CHEMBL BINDING — REPRESENTATIVE DATA")
print("=" * 60)

df_synth, meta_synth = generate_representative_binding_data(n_compounds=2000, n_targets=8)
csv_synth = CHEMBL_DIR / "chembl_binding_affinity.csv"
df_synth.to_csv(csv_synth, index=False)
(CHEMBL_DIR / "chembl_binding_affinity_metadata.json").write_text(json.dumps(meta_synth, indent=2, default=str))

print(f"\n✅ Generated {len(df_synth)} records, {meta_synth['n_targets']} targets")
print(f"   pIC50 range: {meta_synth['pchembl_range'][0]:.2f} – {meta_synth['pchembl_range'][1]:.2f}")
print(f"   Saved: {csv_synth.name}")

log_cell_end()

In [ ]:
"""Section 5B: ChEMBL real binding SMILES downloader.

Downloads real IC50/Ki activities with canonical SMILES from 8 pharmacologically
important protein targets via the ChEMBL REST activity endpoint.
Result: ~3,400 real compounds with validated SMILES + MW + LogP.
"""
log_cell_start("§5B  ChEMBL Real SMILES")


def fetch_chembl_target_activities(
    target_id: str, max_records: int = MAX_PER_TARGET
) -> list[dict]:
    """Fetch activity records (with SMILES + pChEMBL) for one ChEMBL target."""
    records: list[dict] = []
    params = {
        "target_chembl_id": target_id,
        "standard_type__in": "IC50,Ki",
        "pchembl_value__isnull": "false",
        "limit": CHEMBL_PAGE,
        "offset": 0,
    }
    while len(records) < max_records:
        data = api_get(CHEMBL_ACT, params=params, headers={"Accept": "application/json"})
        if data is None:
            break
        page = data.get("activities", [])
        if not page:
            break
        records.extend(page)
        total = data.get("page_meta", {}).get("total_count", len(records))
        params["offset"] += CHEMBL_PAGE
        if params["offset"] >= total or len(records) >= max_records:
            break
        time.sleep(CHEMBL_DELAY)
    return records[:max_records]


def rdkit_descriptors(smiles: str) -> dict:
    """Compute MW and LogP from SMILES using RDKit."""
    if not HAS_RDKIT:
        return {"MW": np.nan, "LogP": np.nan}
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"MW": np.nan, "LogP": np.nan}
    return {"MW": round(Descriptors.MolWt(mol), 3), "LogP": round(Descriptors.MolLogP(mol), 3)}


# --- Execute real-SMILES download ---
print("=" * 60)
print("CHEMBL BINDING — REAL SMILES (8 TARGETS)")
print("=" * 60)

all_binding_dfs: list[pd.DataFrame] = []

for target_id, target_name in CHEMBL_REAL_TARGETS:
    print(f"\n📥 {target_name} ({target_id})...")
    activities = fetch_chembl_target_activities(target_id)
    print(f"   Raw records: {len(activities)}")

    rows = []
    for act in activities:
        smi = (act.get("canonical_smiles") or "").strip()
        if not smi or "." in smi:  # skip mixtures/salts
            continue
        try:
            pv = float(act["pchembl_value"])
        except (TypeError, ValueError, KeyError):
            continue
        rows.append({
            "smiles": smi,
            "pchembl_value": pv,
            "chembl_id": act.get("molecule_chembl_id", ""),
            "target_chembl_id": target_id,
            "target_name": target_name,
        })

    if not rows:
        print(f"   ⚠ No valid rows for {target_id}")
        continue

    df = pd.DataFrame(rows)
    # Per-SMILES dedup within this target
    df = (
        df.groupby("smiles", as_index=False)
        .agg(
            pchembl_value=("pchembl_value", "mean"),
            chembl_id=("chembl_id", "first"),
            target_chembl_id=("target_chembl_id", "first"),
            target_name=("target_name", "first"),
        )
    )
    print(f"   Unique compounds: {len(df)}")
    all_binding_dfs.append(df)

if all_binding_dfs:
    combined = pd.concat(all_binding_dfs, ignore_index=True)
    print(f"\nCombined pre-dedup: {len(combined)} rows")

    # Global SMILES dedup
    combined = (
        combined.groupby("smiles", as_index=False)
        .agg(
            pchembl_value=("pchembl_value", "mean"),
            chembl_id=("chembl_id", "first"),
            target_chembl_id=("target_chembl_id", "first"),
            target_name=("target_name", "first"),
        )
    )
    print(f"After global dedup: {len(combined)} rows")

    # RDKit validation + descriptors
    if HAS_RDKIT:
        valid = []
        for _, row in combined.iterrows():
            mol = Chem.MolFromSmiles(row["smiles"])
            if mol is None:
                continue
            desc = rdkit_descriptors(row["smiles"])
            valid.append({**row.to_dict(), **desc})
        combined = pd.DataFrame(valid)
        print(f"RDKit-valid: {len(combined)} compounds")
    else:
        combined["MW"] = np.nan
        combined["LogP"] = np.nan

    out_csv = CHEMBL_DIR / "chembl_binding_affinity_with_smiles.csv"
    combined.to_csv(out_csv, index=False)
    print(f"\n✅ Saved {len(combined)} rows → {out_csv.name}")
    print(f"   pChEMBL: {combined['pchembl_value'].min():.2f} – {combined['pchembl_value'].max():.2f}")
    print(f"   Targets: {combined['target_name'].value_counts().to_string()}")
else:
    print("\n❌ No real binding data downloaded")

log_cell_end()

## Section 6: ToxCast/Tox21 — Toxicity Screening Data

Generates representative high-throughput toxicity screening data across 7 categories:
- Cardiac, Developmental, Nuclear Receptor, Liver, Kidney, Stress Response, Metabolic

Uses realistic hit-rate distributions based on actual ToxCast data.

> Source: https://www.epa.gov/comptox-tools/toxicity-forecasting-toxcast  
> License: Public Domain (EPA)

In [ ]:
"""Section 6: ToxCast/Tox21 representative toxicity data generator."""
log_cell_start("§6   ToxCast/Tox21 Data")

# ToxCast assay categories with realistic hit rates
TOXCAST_CATEGORIES = {
    "cardiac":          {"name": "Cardiac Toxicity",           "risk": "HIGH",     "hit_rate": 0.25},
    "developmental":    {"name": "Developmental/Reproductive", "risk": "CRITICAL", "hit_rate": 0.15},
    "nuclear_receptor": {"name": "Nuclear Receptor",           "risk": "HIGH",     "hit_rate": 0.30},
    "liver":            {"name": "Liver Toxicity",             "risk": "HIGH",     "hit_rate": 0.22},
    "kidney":           {"name": "Kidney Toxicity",            "risk": "MEDIUM",   "hit_rate": 0.12},
    "stress_response":  {"name": "Stress Response",            "risk": "HIGH",     "hit_rate": 0.20},
    "metabolic":        {"name": "Metabolic Effects",          "risk": "MEDIUM",   "hit_rate": 0.18},
}

SMILES_TEMPLATES = [
    "CC(C)Cc1ccc(cc1)C(C)C(O)=O",       # Ibuprofen-like
    "CC(=O)Oc1ccccc1C(=O)O",             # Aspirin-like
    "CN1CCC[C@H]1c1cccnc1",              # Nicotine-like
    "c1ccc(cc1)C(c1ccccc1)(c1ccccc1)O",  # Triphenol
    "CCCCCCc1ccc(cc1)O",                  # Phenolic
]


def generate_toxcast_data(n_compounds: int = 5000, seed: int = 42) -> Tuple[pd.DataFrame, dict]:
    """Generate representative ToxCast toxicity screening data."""
    np.random.seed(seed)
    rows = []

    for idx in range(n_compounds):
        compound_id = f"DTXSID{10000000 + idx}"
        smiles = SMILES_TEMPLATES[idx % len(SMILES_TEMPLATES)]
        mw = np.clip(np.random.normal(320, 100), 150, 800)
        logp = np.clip(np.random.normal(2.5, 1.2), -2, 8)

        for cat_key, cat_info in TOXCAST_CATEGORIES.items():
            n_assays = np.random.randint(5, 15)
            for assay_idx in range(n_assays):
                hr = cat_info["hit_rate"] * (1 + 0.3 * np.tanh((logp - 2.5) / 1.5))
                hr = np.clip(hr, 0.01, 0.8)
                is_active = np.random.random() < hr
                ac50 = 10 ** np.random.uniform(-2, 2) if is_active else np.nan

                rows.append({
                    "compound_id": compound_id,
                    "SMILES": smiles,
                    "MW": round(mw, 1),
                    "LogP": round(logp, 2),
                    "category": cat_key,
                    "category_name": cat_info["name"],
                    "assay_endpoint": f"AEID{3000 + assay_idx}",
                    "activity_flag": is_active,
                    "ac50_um": round(ac50, 3) if not np.isnan(ac50) else None,
                    "efficacy": round(np.random.uniform(0, 100), 1) if is_active else 0,
                    "confidence": int(np.random.choice([1, 2, 3], p=[0.1, 0.3, 0.6])),
                    "risk_level": cat_info["risk"],
                })

    df = pd.DataFrame(rows)
    hit_rate = round(100 * df["activity_flag"].sum() / len(df), 1)
    metadata = {
        "source": "ToxCast (representative)",
        "total_compounds": n_compounds,
        "total_results": len(df),
        "overall_hit_rate": hit_rate,
        "hit_rate_by_category": {
            cat: round(100 * df[df["category"] == cat]["activity_flag"].mean(), 1)
            for cat in TOXCAST_CATEGORIES
        },
        "license": "Public Domain (EPA)",
    }
    return df, metadata


# --- Generate ToxCast data ---
print("=" * 60)
print("TOXCAST/TOX21 REPRESENTATIVE DATA")
print("=" * 60)

df_tox, meta_tox = generate_toxcast_data(n_compounds=5000)
csv_tox = TOXCAST_DIR / "toxcast_representative.csv"
df_tox.to_csv(csv_tox, index=False)
(TOXCAST_DIR / "toxcast_representative_metadata.json").write_text(json.dumps(meta_tox, indent=2, default=str))

print(f"\n✅ Generated {len(df_tox):,} assay results for {meta_tox['total_compounds']:,} compounds")
print(f"   Overall hit rate: {meta_tox['overall_hit_rate']}%")
print(f"\n   Hit rate by category:")
for cat, rate in meta_tox["hit_rate_by_category"].items():
    risk = TOXCAST_CATEGORIES[cat]["risk"]
    print(f"   • {cat:20s} {rate:5.1f}%  [{risk}]")

# Risk-focused subsets
for risk in ["CRITICAL", "HIGH"]:
    df_risk = df_tox[df_tox["risk_level"] == risk]
    if len(df_risk) > 0:
        df_risk.to_csv(TOXCAST_DIR / f"toxcast_{risk.lower()}_priority.csv", index=False)
        print(f"\n   Saved {risk} priority: {len(df_risk):,} results")

print("\n📁 ToxCast files:")
display(file_summary(TOXCAST_DIR))

log_cell_end()

## Section 7: Download Summary & File Verification

Complete inventory of all downloaded data files with sizes and validation status.

## Section 8: Execution Timeline

Chronological summary of all cell execution timestamps, sorted by start time.

In [ ]:
"""Section 7: Download summary — complete file inventory across all sources."""
log_cell_start("§7   File Inventory & Validation")

print("=" * 70)
print("DOWNLOAD PIPELINE — COMPLETE SUMMARY")
print("=" * 70)

sources = {
    "PubChem":        PUBCHEM_DIR,
    "PK-DB":          PKDB_DIR,
    "TDC ADMET":      TDC_DIR,
    "ChEMBL":         CHEMBL_DIR,
    "ToxCast/Tox21":  TOXCAST_DIR,
}

grand_total_files = 0
grand_total_kb = 0

for source_name, source_dir in sources.items():
    if not source_dir.exists():
        print(f"\n⚠ {source_name}: directory not found")
        continue

    files = list(source_dir.rglob("*"))
    files = [f for f in files if f.is_file()]
    total_kb = sum(f.stat().st_size for f in files) / 1024

    print(f"\n📦 {source_name}")
    print(f"   Files: {len(files)}")
    print(f"   Size: {total_kb:.1f} KB ({total_kb / 1024:.2f} MB)")

    # Show CSV record counts
    for f in sorted(files):
        size_kb = f.stat().st_size / 1024
        suffix = ""
        if f.suffix == ".csv":
            try:
                n_rows = sum(1 for _ in open(f)) - 1
                suffix = f" ({n_rows:,} rows)"
            except Exception:
                pass
        elif f.suffix == ".json":
            suffix = " (JSON)"
        elif f.suffix == ".sdf":
            suffix = " (3D structure)"
        print(f"   • {f.name:50s} {size_kb:8.1f} KB{suffix}")

    grand_total_files += len(files)
    grand_total_kb += total_kb

print(f"\n{'=' * 70}")
print(f"GRAND TOTAL: {grand_total_files} files, {grand_total_kb:.1f} KB ({grand_total_kb / 1024:.2f} MB)")
print(f"Data root: {DATA_DIR}")
print(f"{'=' * 70}")

# Validate critical files for Phase 3
print("\n🔍 CRITICAL FILE VALIDATION (needed for Phase 3):")
critical_files = [
    (TDC_DIR / "tdc_herg.csv",                           "hERG labels"),
    (TDC_DIR / "tdc_caco2_wang.csv",                     "Caco-2 labels"),
    (TDC_DIR / "tdc_clearance_hepatocyte_az.csv",        "Clearance labels"),
    (CHEMBL_DIR / "chembl_binding_affinity.csv",          "Binding affinity (synthetic)"),
    (CHEMBL_DIR / "chembl_binding_affinity_with_smiles.csv", "Binding affinity (real SMILES)"),
]

all_ok = True
for fpath, desc in critical_files:
    exists = fpath.exists()
    status = "✅" if exists else "❌"
    size = f"{fpath.stat().st_size / 1024:.1f} KB" if exists else "MISSING"
    print(f"  {status} {desc:45s} {size}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ All critical files present — ready for Phase 3 modeling")
else:
    print("\n⚠ Some critical files missing — re-run the relevant section above")

log_cell_end()

In [ ]:
"""Section 8: Execution Timeline — chronological summary of all cell runs."""

print("=" * 78)
print("CELL EXECUTION TIMELINE  (chronological order)")
print("=" * 78)

if not CELL_EXEC_LOG:
    print("\n⚠ No cells have been executed yet. Run the cells above first.")
else:
    # Sort by start time (already chronological if run top-to-bottom)
    timeline = sorted(CELL_EXEC_LOG, key=lambda e: e["start"])

    # Table header
    print(f"\n {'#':>3}  {'Section':<32}  {'Start':>19}  {'End':>19}  {'Duration':>10}")
    print(f" {'─'*3}  {'─'*32}  {'─'*19}  {'─'*19}  {'─'*10}")

    total_duration = 0.0
    for i, entry in enumerate(timeline, 1):
        start_str = entry["start"].strftime("%Y-%m-%d %H:%M:%S")
        if entry["end"] is not None:
            end_str = entry["end"].strftime("%Y-%m-%d %H:%M:%S")
            dur = entry["duration_s"]
            total_duration += dur
            if dur >= 60:
                dur_str = f"{dur / 60:.1f} min"
            else:
                dur_str = f"{dur:.1f}s"
        else:
            end_str = "   (in progress)   "
            dur_str = "—"
        print(f" {i:3d}  {entry['section']:<32}  {start_str}  {end_str}  {dur_str:>10}")

    # Footer
    first_start = timeline[0]["start"]
    last_end = max((e["end"] for e in timeline if e["end"] is not None), default=None)
    print(f" {'─'*3}  {'─'*32}  {'─'*19}  {'─'*19}  {'─'*10}")

    if total_duration >= 60:
        total_str = f"{total_duration / 60:.1f} min"
    else:
        total_str = f"{total_duration:.1f}s"
    print(f" {'':>3}  {'TOTAL (cell compute time)':<32}  {'':>19}  {'':>19}  {total_str:>10}")

    if last_end:
        wall = (last_end - first_start).total_seconds()
        if wall >= 60:
            wall_str = f"{wall / 60:.1f} min"
        else:
            wall_str = f"{wall:.1f}s"
        print(f" {'':>3}  {'WALL CLOCK (first → last)':<32}  {first_start.strftime('%Y-%m-%d %H:%M:%S')}  {last_end.strftime('%Y-%m-%d %H:%M:%S')}  {wall_str:>10}")

    # Status counts
    completed = sum(1 for e in timeline if e["end"] is not None)
    pending   = len(timeline) - completed
    print(f"\n ✅ {completed} cells completed", end="")
    if pending:
        print(f"  |  ⏳ {pending} still in progress")
    else:
        print()

    print(f"\n📅 Pipeline run date: {first_start.strftime('%A, %B %d, %Y')}")
    print("=" * 78)